# 13 — JSON, CSV, and Data Pipeline Design

Real programs move data between text files, APIs, and internal records. Build reliable ETL (extract, transform, load) steps that parse values, validate assumptions, and keep transformations independent of storage.

*Intermediate · 75–120 min*

## You will learn

- Serialize and deserialize JSON safely
- Read CSV as strings and convert numeric columns
- Build pure validation and transformation functions
- Explain schema checks and partial failure policies

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. JSON objects and Python values

JSON encodes objects, arrays, strings, numbers, booleans, and null. json.loads parses a JSON string into Python structures; json.dumps serializes values. JSON does not directly encode arbitrary Python classes, sets, or datetime objects without conversion.

In [ ]:
import json

raw = '{"customer": "Ada", "items": [2, 3], "active": true}'
record = json.loads(raw)
print(record["customer"], sum(record["items"]))
print(json.dumps(record, sort_keys=True, indent=2))

**Think about it:** What Python value corresponds to JSON null?

### 2. Validating external records

Parsing JSON only validates its syntax, not the business meaning. Confirm keys exist, check types and constraints, and decide whether an invalid row should be rejected or skipped. Use clearly named functions to separate these choices.

In [ ]:
def parse_product(record):
    if not isinstance(record, dict):
        raise ValueError("expected an object")
    name = record.get("name")
    price = record.get("price")
    if not isinstance(name, str) or not name.strip():
        raise ValueError("name must be nonempty text")
    if isinstance(price, bool) or not isinstance(price, (int, float)) or price < 0:
        raise ValueError("price must be a nonnegative number")
    return {"name": name.strip(), "price": float(price)}

print(parse_product({"name": " Pen ", "price": 2.5}))

**Think about it:** Why should true not silently be accepted as a numeric price even though bool subclasses int?

### 3. CSV conversion and schema

CSV has no built-in types. DictReader yields strings for each field, so you must convert columns explicitly. Always supply newline='' when using csv and consistent encoding. The demo uses a string stream to avoid writing repository files.

In [ ]:
import csv
from io import StringIO

raw_csv = "sku,quantity,unit_price\nP1,3,4.50\nP2,2,8.25\n"
rows = list(csv.DictReader(StringIO(raw_csv)))
totals = [int(row["quantity"]) * float(row["unit_price"]) for row in rows]
print(rows)
print(totals, sum(totals))

**Think about it:** What happens if quantity is missing or contains invalid text?

### 4. Composable ETL stages

A pipeline becomes easier to test when extract, transform, and load stages are separate. Use iterables or generators for scalable processing, and build a small report describing which records succeeded or failed.

In [ ]:
def validate_rows(rows):
    valid, errors = [], []
    for index, row in enumerate(rows, start=1):
        try:
            valid.append(parse_product(row))
        except ValueError as exc:
            errors.append({"index": index, "reason": str(exc)})
    return valid, errors

source_rows = [
    {"name": "Book", "price": 12},
    {"name": "", "price": 4},
    {"name": "Pen", "price": 1.5},
]
valid, errors = validate_rows(source_rows)
print(valid)
print(errors)

**Think about it:** How would you preserve the row's original identifier when reporting errors?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/03_data_quality_and_testing_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/13', {1: '801737b087923c51', 2: '18641c2fbb766610', 3: '003313b5b3639365', 4: '40ccd4c91a85cfcd'})

**Exercise 1.** Parse product JSON and report entries without a numeric price.

In [ ]:
import json
raw = '[{"sku": "A", "price": 2.5}, {"sku": "B", "price": "free"}, {"sku": "C", "price": 4}]'
valid, invalid = [], []
for item in json.___(raw):
    if isinstance(item["price"], ___):
        valid.append(item["sku"])
    else:
        invalid.append(item["sku"])
check(1, (valid, invalid))

**Exercise 2.** CSV values are text: convert them, then compute a total.

In [ ]:
import csv, io
text = "item,qty,price\npen,2,1.50\nink,1,4.00\n"
total = sum(___ * float(row["price"]) for row in csv.DictReader(io.StringIO(text)))
check(2, total)

**Exercise 3.** Write a JSON report with a summary and the number of rejected records.

In [ ]:
import json
accepted, rejected = ["A", "C"], ["B"]
report = json.___({"accepted": len(accepted), "rejected": ___}, sort_keys=True)
check(3, report)

**Exercise 4.** A CSV round trip in memory with `StringIO` and `DictWriter`.

In [ ]:
import csv, io
rows = [{"name": "ana", "score": "90"}, {"name": "ben", "score": "75"}]
buffer = io.StringIO()
writer = csv.DictWriter(buffer, fieldnames=["name", "score"])
writer.writeheader()
writer.writerows(rows)
buffer.seek(___)
back = list(csv.___(buffer))
check(4, back == rows)

## Common mistakes

- Syntactically correct JSON can still contain invalid domain values.
- CSV rows have text fields; validate conversions before calculations.
- Never use eval() to parse JSON or configuration.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Which converts a JSON string into Python objects?

   - **a)** eval
   - **b)** json.loads
   - **c)** json.dumps

**2.** Why is eval() unsafe for parsing data?

   - **a)** It doesn't handle numbers
   - **b)** It can execute arbitrary code
   - **c)** It's slow

**3.** A good partial-failure policy for bad CSV rows is…

   - **a)** Collect and report rejected rows with their line numbers
   - **b)** Crash on the first bad row, always
   - **c)** Ignore them silently

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/13', {1: '6bceaacdd613', 2: 'af221cbe4dc0', 3: '8bb1e89db1e9'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [14 — Unit Testing, Fixtures, and Mocking](14_unittest_and_mocking.ipynb)